In [1]:
import json
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq
from tqdm import tqdm

In [3]:
ip_file = "metadata/arxiv-metadata-oai-snapshot.json"
op_file = "metadata/arxiv_filtered_master.parquet"
target_cats = { 'cs.AI','cs.LG','cs.NE','cs.CL','cs.CV','stat.ML','cs.IR','cs.GT'}

In [7]:
i=0
with open(ip_file,'r') as f:
    for line in f:
        doc= json.loads(line)
        i+=1
        if i==10:
            break

        print(doc)

{'id': '0704.0001', 'submitter': 'Pavel Nadolsky', 'authors': "C. Bal\\'azs, E. L. Berger, P. M. Nadolsky, C.-P. Yuan", 'title': 'Calculation of prompt diphoton production cross sections at Tevatron and\n  LHC energies', 'comments': '37 pages, 15 figures; published version', 'journal-ref': 'Phys.Rev.D76:013009,2007', 'doi': '10.1103/PhysRevD.76.013009', 'report-no': 'ANL-HEP-PR-07-12', 'categories': 'hep-ph', 'license': None, 'abstract': '  A fully differential calculation in perturbative quantum chromodynamics is\npresented for the production of massive photon pairs at hadron colliders. All\nnext-to-leading order perturbative contributions from quark-antiquark,\ngluon-(anti)quark, and gluon-gluon subprocesses are included, as well as\nall-orders resummation of initial-state gluon radiation valid at\nnext-to-next-to-leading logarithmic accuracy. The region of phase space is\nspecified in which the calculation is most reliable. Good agreement is\ndemonstrated with data from the Fermilab

In [17]:
def filter_arxiv_dump():
    print(f"reading {ip_file}")

    batch_data=[]
    batch_size = 50000
    total_papers=0
    kept_papers=0

    schema = pa.schema([
    ('id',pa.string()),
    ('title', pa.string()),
    ('abstract',pa.string()),
    ('year',pa.int32()),
    ('categories',pa.string()),
    ('version',pa.string())
    ])

    writer = None

    with open(ip_file,'r') as f:
        for line in tqdm(f, desc='Processing Lines'):
            try:
                doc = json.loads(line)
                total_papers +=1

                doc_cats = set(doc['categories'].split(' '))
                # print(doc_cats)
                # if total_papers == 10:
                #     break
                if not doc_cats.isdisjoint(target_cats):

                    try:
                        v1_date = doc['versions'][0]['created']
                        year = int(v1_date.split(' ')[3])
                    except:
                        year = 0

                    batch_data.append({
                    'id':doc['id'],
                    'title': doc['title'].replace('\n', ' ').strip(),
                    'abstract': doc['abstract'].replace('\n', ' ').strip(),
                    'categories': doc['categories'],
                    'year': year,
                    'versions': json.dumps(doc['versions'])
                    })
                    kept_papers+=1

                if len(batch_data)>=batch_size:
                    table = pa.Table.from_pylist(batch_data,schema=schema)
                    if writer is None:
                        writer = pq.ParquetWriter(op_file,schema)
                    writer.write_table(table)
                    batch_data=[] #clear ram

            except Exception as e:
                print(e)
                continue 

    if batch_data:
        table = pa.Table.from_pylist(batch_data, schema=schema)
        if writer is None:
            writer = pq.ParquetWriter(OUTPUT_FILE, schema)
        writer.write_table(table)
        
    if writer:
        writer.close()

    print(f"\n✅ Done! Scanned {total_papers} papers.")
    print(f"🎯 Kept {kept_papers} relevant papers.")
    print(f"💾 Saved to {op_file}")
        
        

                

In [18]:
filter_arxiv_dump()

reading metadata/arxiv-metadata-oai-snapshot.json


Processing Lines: 2938427it [00:24, 122036.10it/s]



✅ Done! Scanned 2938427 papers.
🎯 Kept 542924 relevant papers.
💾 Saved to arxiv_filtered_master.parquet


In [19]:
import pandas as pd

df_check = pd.read_parquet("arxiv_filtered_master.parquet")
print("Year Distribution:")
print(df_check['year'].value_counts().sort_index())

# Check for the missing QLoRA paper by its known ID
print("\nChecking for QLoRA (2305.14314):")
print(df_check[df_check['id'] == '2305.14314'])

Year Distribution:
year
1993         6
1994       237
1995       252
1996       230
1997       184
1998       160
1999       111
2000       283
2001       168
2002       263
2003       262
2004       306
2005       326
2006       403
2007       582
2008       859
2009      1250
2010      1740
2011      2403
2012      3934
2013      5142
2014      5538
2015      7265
2016     10907
2017     15968
2018     24080
2019     34248
2020     45677
2021     50270
2022     55366
2023     69302
2024     89604
2025    109893
2026      5705
Name: count, dtype: int64

Checking for QLoRA (2305.14314):
                id                                          title  \
288962  2305.14314  QLoRA: Efficient Finetuning of Quantized LLMs   

                                                 abstract  year categories  \
288962  We present QLoRA, an efficient finetuning appr...  2023      cs.LG   

       version  
288962    None  


In [23]:
df_check['id']

0                0704.0047
1                0704.0050
2                0704.0304
3                0704.0671
4                0704.0954
                ...       
542919    quant-ph/0609117
542920    quant-ph/0611234
542921    quant-ph/0702072
542922    quant-ph/9802028
542923    quant-ph/9907009
Name: id, Length: 542924, dtype: object

In [3]:
file = "arxiv_filtered_master.parquet"

In [4]:
df = pd.read_parquet(file)

In [8]:
with pd.option_context('display.max_colwidth', None):
    display(df.sample(n=20))

,id,title,abstract,year,categories,version
485842,2507.23309,PriorFusion: Unified Integration of Priors for Robust Road Perception in Autonomous Driving,"With the growing interest in autonomous driving, there is an increasing demand for accurate and reliable road perception technologies. In complex environments without high-definition map support, autonomous vehicles must independently interpret their surroundings to ensure safe and robust decision-making. However, these scenarios pose significant challenges due to the large number, complex geometries, and frequent occlusions of road elements. A key limitation of existing approaches lies in their insufficient exploitation of the structured priors inherently present in road elements, resulting in irregular, inaccurate predictions. To address this, we propose PriorFusion, a unified framework that effectively integrates semantic, geometric, and generative priors to enhance road element perception. We introduce an instance-aware attention mechanism guided by shape-prior features, then construct a data-driven shape template space that encodes low-dimensional representations of road elements, enabling clustering to generate anchor points as reference priors. We design a diffusion-based framework that leverages these prior anchors to generate accurate and complete predictions. Experiments on large-scale autonomous driving datasets demonstrate that our method significantly improves perception accuracy, particularly under challenging conditions. Visualization results further confirm that our approach produces more accurate, regular, and coherent predictions of road elements.",2025,cs.CV,None
157193,2012.07121,Deliberative and Conceptual Inference in Service Robots,"Service robots need to reason to support people in daily life situations. Reasoning is an expensive resource that should be used on demand whenever the expectations of the robot do not match the situation of the world and the execution of the task is broken down; in such scenarios the robot must perform the common sense daily life inference cycle consisting on diagnosing what happened, deciding what to do about it, and inducing and executing a plan, recurring in such behavior until the service task can be resumed. Here we examine two strategies to implement this cycle: (1) a pipe-line strategy involving abduction, decision-making and planning, which we call deliberative inference and (2) the use of the knowledge and preferences stored in the robot's knowledge-base, which we call conceptual inference. The former involves an explicit definition of a problem space that is explored through heuristic search, and the latter is based on conceptual knowledge including the human user preferences, and its representation requires a non-monotonic knowledge-based system. We compare the strengths and limitations of both approaches. We also describe a service robot conceptual model and architecture capable of supporting the daily life inference cycle during the execution of a robotics service task. The model is centered in the declarative specification and interpretation of robot's communication and task structure. We also show the implementation of this framework in the fully autonomous robot Golem-III. The framework is illustrated with two demonstration scenarios.",2020,cs.RO cs.AI,None
297639,2306.17256,Could Small Language Models Serve as Recommenders? Towards Data-centric Cold-start Recommendations,"Recommendation systems help users find matched items based on their previous behaviors. Personalized recommendation becomes challenging in the absence of historical user-item interactions, a practical problem for startups known as the system cold-start recommendation. While existing research addresses cold-start issues for either users or items, we still lack solutions for system cold-start scenarios. To tackle the problem, we propose PromptRec, a simple but effective approach based on in-context learning of language models, where we trans

In [9]:
jn = pd.read_json('test_retrieval_set.json')

In [10]:
jn

,query,target_id,type
0,PriorFusion autonomous driving road perception...,2507.23309,keyword
1,How can we use diffusion models and shape prio...,2507.23309,semantic
2,unifying semantic geometric and generative pri...,2507.23309,conceptual
3,deliberative conceptual inference service robo...,2012.07121,keyword
4,What are the differences between deliberative ...,2012.07121,semantic
...,...,...,...
85,How does jointly conditioning on left and righ...,1810.04805,semantic
86,creating task-agnostic state-of-the-art repres...,1810.04805,conceptual
87,Vision Transformer ViT image patches classific...,2010.11929,keyword
88,Can we use a pure transformer architecture on ...,2010.11929,semantic
